# Dataset Statistics

Scans the raw AgrI Challenge dataset, extracts per-image metadata (dimensions, device, perceptual hash), and saves a summary CSV used by the downstream notebooks.

**Outputs:**
- `image_dataset_summary_with_hash.csv` (or `..._no_hash.csv`) -- one row per image
- `duplicates.csv` -- potential duplicate images identified by perceptual hash


In [9]:
# Install required libraries directly from the notebook
%pip install pandas Pillow pillow-heif imagehash

# Import the libraries
import os
import json
import pandas as pd
from PIL import Image, ExifTags
from pillow_heif import register_heif_opener
import imagehash
import warnings

# Ignore warnings that can clutter the output, like from malformed EXIF data
warnings.filterwarnings("ignore")

print("✅ Libraries installed and imported successfully.")

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
✅ Libraries installed and imported successfully.


In [10]:
# Enable reading of .heic and .heif files
register_heif_opener()

# --- IMPORTANT ---
# Change this to the full path of your dataset's root folder
DATASET_ROOT = './Agrichallenge_dataset/AgrI_Dataset/Teams_space/'

# To scan only specific teams, list their folder names here.
# Example: TEAMS_TO_SCAN = ['AiGro', 'TeamAlpha']
# To scan ALL teams, leave the list empty.
TEAMS_TO_SCAN = []

# --- Parameter for performance ---
# Set to True to calculate a visual hash for finding duplicates.
# Set to False to skip this step for a much faster scan.
CALCULATE_HASH = True

# Define the image file extensions to look for
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.heic', '.bmp', '.tiff', '.tif')

# List to hold the data for each image
image_data = []

print("Configuration set.")

Configuration set.


In [ ]:
if not os.path.isdir(DATASET_ROOT):
    print(f"Error: The directory '{DATASET_ROOT}' does not exist. Please check the DATASET_ROOT path in the configuration cell.")
else:
    print("Starting image scan...")
    if TEAMS_TO_SCAN:
        print(f"Scanning selected teams: {', '.join(TEAMS_TO_SCAN)}")
    else:
        print("Scanning all teams.")

    if not CALCULATE_HASH:
        print("Note: image hashing is disabled (CALCULATE_HASH=False).")

    announced_classes = set()

    for dirpath, dirnames, filenames in os.walk(DATASET_ROOT):
        if dirpath == DATASET_ROOT and TEAMS_TO_SCAN:
            dirnames[:] = [d for d in dirnames if d in TEAMS_TO_SCAN]
            found_teams = set(dirnames)
            missing_teams = set(TEAMS_TO_SCAN) - found_teams
            if missing_teams:
                print(f"Warning: the following specified teams were not found and will be skipped: {', '.join(missing_teams)}")

        relative_path = os.path.relpath(dirpath, DATASET_ROOT)

        if relative_path == '.':
            continue

        path_parts = relative_path.split(os.sep)

        if len(path_parts) >= 2:
            team_name = path_parts[0]
            class_name = path_parts[1]

            class_id = (team_name, class_name)
            if class_id not in announced_classes:
                print(f"\n-> Scanning: {team_name} / {class_name}")
                announced_classes.add(class_id)

            for file_name in filenames:
                if file_name.lower().endswith(IMAGE_EXTENSIONS):
                    file_path = os.path.join(dirpath, file_name)

                    info = {
                        'image_path': file_path, 'image_name': file_name, 'team': team_name,
                        'class': class_name, 'extension': os.path.splitext(file_name)[1].lower(),
                        'image_hash': 'skipped', 'size_mb': round(os.path.getsize(file_path) / (1024 * 1024), 2),
                        'width': 'unknown', 'height': 'unknown', 'device': 'unknown',
                        'metadata': '{}', 'corrupted': 'no'
                    }

                    try:
                        with Image.open(file_path) as img:
                            if CALCULATE_HASH:
                                info['image_hash'] = str(imagehash.phash(img))

                            info['width'] = img.width
                            info['height'] = img.height

                            exif_data = img.getexif()
                            if exif_data:
                                decoded_exif = {ExifTags.TAGS.get(tag, tag): val for tag, val in exif_data.items()}

                                make_val = decoded_exif.get('Make')
                                model_val = decoded_exif.get('Model')

                                make = make_val.decode(errors='ignore').strip() if isinstance(make_val, bytes) else str(make_val or '').strip()
                                model = model_val.decode(errors='ignore').strip() if isinstance(model_val, bytes) else str(model_val or '').strip()

                                if make or model:
                                    info['device'] = f"{make} {model}".strip()

                                decoded_metadata = {}
                                for key, val in decoded_exif.items():
                                    if isinstance(val, bytes):
                                        decoded_metadata[key] = val.decode('utf-8', errors='replace').strip('\x00')
                                    else:
                                        decoded_metadata[key] = val
                                decoded_metadata.pop('MakerNote', None)
                                info['metadata'] = json.dumps(decoded_metadata, default=str)

                    except Exception as e:
                        print(f"  Warning: could not process '{file_name}'. Marked as corrupted. Error: {e}")
                        info['corrupted'] = 'yes'

                    image_data.append(info)

    print("\nScan complete.")

df = pd.DataFrame(image_data)


🚀 Starting image scan...
🎯 Scanning all teams.

-> Now scanning class: AiGro / frene

-> Now scanning class: AiGro / caroubier

-> Now scanning class: AiGro / chene

-> Now scanning class: AiGro / tipu

-> Now scanning class: AiGro / faux_poivrier 
    [!] Warning: Could not process file '20240625_094625.jpg'. Marked as corrupted. Error: cannot identify image file './Agrichallenge_dataset/AgrI_Dataset/Teams_space/AiGro/faux_poivrier /20240625_094625.jpg'

-> Now scanning class: AiGro / pistachier 

-> Now scanning class: the Neural Ninjas / pistachier

-> Now scanning class: the Neural Ninjas / caroubier

-> Now scanning class: the Neural Ninjas / tipu

-> Now scanning class: the Neural Ninjas / frene

-> Now scanning class: the Neural Ninjas / faux_poivrier

-> Now scanning class: the Neural Ninjas / chene

-> Now scanning class: AI-4o / frene

-> Now scanning class: AI-4o / caroubier

-> Now scanning class: AI-4o / tipu

-> Now scanning class: AI-4o / chene

-> Now scanning class: AI

In [12]:
# Display the first 10 rows of the DataFrame
print("Displaying the first 10 rows of the summary table:")
df.head(10)

Displaying the first 10 rows of the summary table:


,image_path,image_name,team,class,extension,image_hash,size_mb,width,height,device,metadata,corrupted
0,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_105058.jpg,AiGro,frene,.jpg,8739c538cf943b85,17.05,8992,8992,samsung                         SM-M536B      ...,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Ima...",no
1,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_105250.jpg,AiGro,frene,.jpg,c3c53c95e3b441bc,16.99,8992,8992,samsung SM-M536B,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Res...",no
2,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_103917.jpg,AiGro,frene,.jpg,f5a32a4e8e394987,24.68,8992,8992,samsung SM-M536B,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Res...",no
3,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_105013.jpg,AiGro,frene,.jpg,a09ad398e179697c,22.73,8992,8992,samsung SM-M536B,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Res...",no
4,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_104831.jpg,AiGro,frene,.jpg,a5a00a0f57f0d3db,24.40,8992,8992,samsung SM-M536B,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Res...",no
5,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_102653.jpg,AiGro,frene,.jpg,e2c48e96353a723e,18.47,8992,8992,samsung                         SM-M536B      ...,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Ima...",no
6,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG_2490.heic,AiGro,frene,.heic,a2d9c4d77a04f287,1.29,4284,4284,Apple iPhone 15,"{""TileWidth"": 640, ""TileLength"": 896, ""Resolut...",no
7,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_103120.jpg,AiGro,frene,.jpg,cccfa81ea44fe588,22.77,8992,8992,samsung SM-M536B,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Res...",no
8,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,20240625_103019.jpg,AiGro,frene,.jpg,add8e502c53aed92,23.92,8992,8992,samsung                         SM-M536B      ...,"{""ImageWidth"": 8992, ""ImageLength"": 8992, ""Ima...",no
9,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG_2484.heic,AiGro,frene,.heic,f0ecc00dc12efaf4,1.22,4284,4284,Apple iPhone 15,"{""TileWidth"": 640, ""TileLength"": 896, ""Resolut...",no


In [13]:
if not df.empty:
    print("📊 Dataset Summary Statistics 📊")
    print("--------------------------------")

    # Total number of images
    print(f"Total Images Scanned: {len(df)}")
    
    # Number of images per team
    print("\nImages per Team:")
    print(df['team'].value_counts().to_string())

    # Number of images per class
    print("\nImages per Class:")
    print(df['class'].value_counts().to_string())

    # File size statistics
    print("\nFile Size (MB) Statistics:")
    print(f"  Average: {df['size_mb'].mean():.2f} MB")
    print(f"  Min: {df['size_mb'].min():.2f} MB")
    print(f"  Max: {df['size_mb'].max():.2f} MB")

    # Image dimension statistics
    # Convert to numeric, coercing errors for 'unknown' values
    numeric_width = pd.to_numeric(df['width'], errors='coerce')
    numeric_height = pd.to_numeric(df['height'], errors='coerce')
    print("\nImage Dimensions (Width × Height) Statistics:")
    print(f"  Average: {numeric_width.mean():.0f} × {numeric_height.mean():.0f}")
    print(f"  Min: {numeric_width.min():.0f} × {numeric_height.min():.0f}")
    print(f"  Max: {numeric_width.max():.0f} × {numeric_height.max():.0f}")

    # Device statistics
    print("\nTop 5 Devices Used:")
    print(df['device'].value_counts().nlargest(5).to_string())
    
    # Corrupted file count
    corrupted_count = df[df['corrupted'] == 'yes'].shape[0]
    print(f"\nCorrupted or Unreadable Files: {corrupted_count}")
    
else:
    print("No data to summarize. Please check your folder path and ensure it contains images.")

📊 Dataset Summary Statistics 📊
--------------------------------
Total Images Scanned: 47369

Images per Team:
team
AI-4o                  7344
PLT                    6556
SMART AGRICULTURES     6321
condimenteum           5504
CACTUS                 3801
GreenAI                3785
AiGro                  3634
RUSTICUS               3447
Scorpions              2795
CHAJARA                2540
the Neural Ninjas      1642

Images per Class:
class
tipu              8034
chene             7846
frene             7453
caroubier         7328
faux_poivrier     6860
pistachier        6255
pistachier        1263
faux_poivrier     1205
chenes             625
frenes             500

File Size (MB) Statistics:
  Average: 2.57 MB
  Min: 0.00 MB
  Max: 32.93 MB

Image Dimensions (Width × Height) Statistics:
  Average: 2709 × 2738
  Min: 250 × 250
  Max: 8992 × 8992

Top 5 Devices Used:
device
unknown                  10986
Apple iPhone 11           5591
OPPO OPPO Reno5           3914
samsung Galaxy A5

In [14]:
# Dynamically set the filename based on whether hashing was enabled
if CALCULATE_HASH:
    output_csv_path = 'image_dataset_summary_with_hash.csv'
else:
    output_csv_path = 'image_dataset_summary_no_hash.csv'

# Save the DataFrame to the determined CSV file path
df.to_csv(output_csv_path, index=False)

print(f"✅ Summary table saved successfully to '{output_csv_path}'")

✅ Summary table saved successfully to 'image_dataset_summary_with_hash.csv'


In [15]:
#Read the csv file 
csv_file = "image_dataset_summary_with_hash.csv"
df = pd.read_csv(csv_file)

In [16]:
# First, check if hashing was enabled during the scan
if not CALCULATE_HASH:
    print("⚠️ Hashing was disabled during the scan.")
    print("   To check for duplicates, set CALCULATE_HASH = True in Cell 2 and re-run the notebook.")
else:
    print("🔎 Checking for potential duplicate images based on image hash...")

    # Find all rows where the image_hash is duplicated
    duplicates = df[df.duplicated(subset=['image_hash'], keep=False)]

    # Filter out corrupted images or where hashing was skipped/failed
    duplicates = duplicates[~duplicates['image_hash'].isin(['skipped', 'unknown'])]

    if not duplicates.empty:
        print(f"\nFound {len(duplicates)} potential duplicates across {duplicates['image_hash'].nunique()} unique images.")
        
        # Sort by hash to group duplicates together
        duplicates_sorted = duplicates.sort_values(by='image_hash')
        
        # Save to CSV
        output_file = "duplicates.csv"
        duplicates_sorted.to_csv(output_file, index=False)
        print(f"📁 Duplicate list saved to: {output_file}")
        
        # Optionally display in notebook
        display(duplicates_sorted)
    else:
        print("\n✅ No duplicate images found.")

🔎 Checking for potential duplicate images based on image hash...

✅ No duplicate images found.
